# 1. Data checks

The data comes from Kaggle, used in a competition on building a credit risk scoring algorithm.

The dataset consists of two files: a **training set** and a **test set**.

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [2]:
df_train = pd.read_csv('../data/cs-training.csv', index_col=0)
df_test  = pd.read_csv('../data/cs-test.csv', index_col=0)

In [3]:
len(df_train)

150000

In [4]:
len(df_test)

101503

## Data Quality

### NULL and duplicates

In [5]:
df_train.isnull().sum()

SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

In [6]:
print(
    "MonthlyIncome NULL ratio :", round(df_train["MonthlyIncome"].isnull().mean(), 4),
    "\nNumberOfDependents NULL ratio :", round(df_train["NumberOfDependents"].isnull().mean(), 4)
)

MonthlyIncome NULL ratio : 0.1982 
NumberOfDependents NULL ratio : 0.0262


In [7]:
df_train.duplicated().sum()/len(df_train)

np.float64(0.00406)

### Inconsistencies

In [8]:
df_train.describe()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,"150,000.00","150,000.00","150,000.00","150,000.00","150,000.00","120,269.00","150,000.00","150,000.00","150,000.00","150,000.00","146,076.00"
mean,0.07,6.05,52.30,0.42,353.01,"6,670.22",8.45,0.27,1.02,0.24,0.76
std,0.25,249.76,14.77,4.19,"2,037.82","14,384.67",5.15,4.17,1.13,4.16,1.12
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.03,41.00,0.00,0.18,"3,400.00",5.00,0.00,0.00,0.00,0.00
50%,0.00,0.15,52.00,0.00,0.37,"5,400.00",8.00,0.00,1.00,0.00,0.00
75%,0.00,0.56,63.00,0.00,0.87,"8,249.00",11.00,0.00,2.00,0.00,1.00
max,1.00,"50,708.00",109.00,98.00,"329,664.00","3,008,750.00",58.00,98.00,54.00,98.00,20.00


In [9]:
# Debt Ratio
income_null = df_train["MonthlyIncome"].isnull() | (df_train["MonthlyIncome"] == 0)

print("===== DebtRatio =====")
print(f"  MonthlyIncome populated -> median {df_train.loc[~income_null, 'DebtRatio'].median():.2f} "
      f"| mean {df_train.loc[~income_null, 'DebtRatio'].mean():.0f}")
print(f"  MonthlyIncome NULL or 0 -> median {df_train.loc[income_null, 'DebtRatio'].median():.2f} "
      f"| mean {df_train.loc[income_null, 'DebtRatio'].mean():.0f}")

===== DebtRatio =====
  MonthlyIncome populated -> median 0.29 | mean 5
  MonthlyIncome NULL or 0 -> median 1144.00 | mean 1668


In [10]:
# Age
print("===== TRAIN : age =====")
print(f"  age < 18  : {(df_train['age'] < 18).sum()}")
print(f"  min / max : {df_train['age'].min()} / {df_train['age'].max()}")

===== TRAIN : age =====
  age < 18  : 1
  min / max : 0 / 109


In [11]:
s = df_train["NumberOfDependents"]
print("===== TRAIN : NumberOfDependents =====")
print(f"  max : {s.max()}")
print(s.value_counts().sort_index().tail(6))

===== TRAIN : NumberOfDependents =====
  max : 20.0
NumberOfDependents
7.00     51
8.00     24
9.00      5
10.00     5
13.00     1
20.00     1
Name: count, dtype: int64


In [12]:
# Past-due counters
counter_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
df_train["delinquency_sentinel"] = (df_train[counter_cols] >= 90).any(axis=1).astype(int)
df_train[counter_cols] = df_train[counter_cols].mask(df_train[counter_cols] >= 90)

n_flagged = df_train["delinquency_sentinel"].sum()
print(f"Sentinel rows flagged: {n_flagged:,} ({n_flagged / len(df_train):.2%} of population)")
print(f"Default rate — sentinel: {df_train.loc[df_train['delinquency_sentinel'] == 1, 'SeriousDlqin2yrs'].mean():.1%} "
      f"vs baseline {df_train['SeriousDlqin2yrs'].mean():.1%}")

Sentinel rows flagged: 269 (0.18% of population)
Default rate — sentinel: 54.6% vs baseline 6.7%


## Cleaning

In [13]:
# TRAIN cleaning
df_train_clean = df_train.copy()
income_invalid = df_train_clean["MonthlyIncome"].isnull() | (df_train_clean["MonthlyIncome"] == 0)
df_train_clean.loc[income_invalid, "DebtRatio"] = np.nan

past_due_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
df_train_clean[past_due_cols] = df_train_clean[past_due_cols].mask(df_train_clean[past_due_cols] >= 90)

df_train_clean = df_train_clean[~((df_train_clean["age"] < 18) | (df_train_clean["NumberOfDependents"] > 15))]

In [14]:
# Export
df_train_clean.to_csv("../data/train_clean.csv")